# 🐺 WolfPicture — Printify Baskı Analiz Stüdyosu 2026

Bu sürüm **arka plan silmez, upscale yapmaz ve görselin piksellerini değiştirmez**.

Sen AI ile hazırladığın son görseli doğrudan yüklersin. Notebook şu işleri yapar:

1. Ortamı güvenli biçimde hazırlar  
2. Görseli yükler  
3. Teknik baskı analizini yapar  
4. Baskıya uygunluk kararı verir  
5. Uygun tişört renklerini sıralar  
6. Minimum ve önerilen baskı genişliğini hesaplar  
7. Açık ve koyu kumaş risklerini açıklar  
8. Tişört mockupları oluşturur  
9. Son kalite özetini gösterir  
10. CSV ve JSON raporu üretir  
11. Bütün sonuçları ZIP olarak indirir  

> **Önemli:** Bu notebook yalnızca analiz ve ön izleme yapar. Orijinal görsele dokunmaz.

## 1️⃣ Güvenli kurulum

Bu hücre gerekli paketleri uyumlu sürümlerle kontrol eder.  
Kurulum gerekirse Colab oturumu otomatik olarak yeniden başlar. Yeniden bağlandıktan sonra aynı hücreyi bir kez daha çalıştır.

In [ ]:
#@title 1️⃣ Güvenli Kurulum
import sys, subprocess, os
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "Pillow": "11.3.0",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "matplotlib": "3.10.0",
    "opencv-python-headless": "4.10.0.84"
}

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

missing_or_wrong = {
    pkg: wanted for pkg, wanted in REQUIRED.items()
    if installed_version(pkg) != wanted
}

if missing_or_wrong:
    print("🔧 Uyumlu paketler kuruluyor...")
    packages = [f"{pkg}=={ver}" for pkg, ver in missing_or_wrong.items()]
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--upgrade", "--force-reinstall"] + packages,
        text=True, capture_output=True
    )
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("Paket kurulumu tamamlanamadı.")

    print("✅ Kurulum tamamlandı.")
    print("🔄 Colab oturumu şimdi otomatik yeniden başlayacak.")
    os.kill(os.getpid(), 9)
else:
    print("✅ Bütün paketler doğru sürümde. 2. adıma geçebilirsin.")

🔧 Uyumlu paketler kuruluyor...


## 2️⃣ Ortam doğrulama

Pillow, OpenCV, NumPy ve Pandas paketlerinin sorunsuz çalıştığını kontrol eder.

In [ ]:
#@title 2️⃣ Ortam Doğrulama
import sys
import PIL
import cv2
import numpy as np
import pandas as pd
import matplotlib
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display

print("🐍 Python :", sys.version.split()[0])
print("🖼️ Pillow:", PIL.__version__)
print("🔍 OpenCV:", cv2.__version__)
print("🧮 NumPy :", np.__version__)
print("📊 Pandas:", pd.__version__)
print("📈 Matplotlib:", matplotlib.__version__)
print("\n✅ Ortam sorunsuz çalışıyor.")

## 3️⃣ Analiz ayarları

Varsayılan değerler çoğu DTG/DTF tişört tasarımı için uygundur.

- **Hedef DPI:** Teknik baskı hesabında kullanılır.
- **Varsayılan önerilen genişlik:** Detay analizi güvenli bulursa hedef olarak kullanılır.
- **Maksimum dosya boyutu:** Yükleme kontrolü için kullanılır.
- **Mockup:** En iyi tişört renklerinde ön izleme üretir.

In [ ]:
#@title 3️⃣ Analiz Ayarları
from pathlib import Path

target_dpi = 300 #@param {type:"integer"}
default_recommended_width_cm = 30 #@param {type:"integer"}
max_file_size_mb = 100 #@param {type:"integer"}
create_mockups = True #@param {type:"boolean"}
mockup_color_count = 4 #@param [2, 3, 4, 5]

WORKDIR = Path("/content/WolfPicture_Analyzer")
INPUT_DIR = WORKDIR / "01_input"
MOCKUP_DIR = WORKDIR / "02_mockups"
REPORT_DIR = WORKDIR / "03_reports"

for folder in [INPUT_DIR, MOCKUP_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Ayarlar kaydedildi.")
print("📁 Çalışma klasörü:", WORKDIR)
print("🖨️ Hedef DPI:", target_dpi)
print("📏 Varsayılan önerilen genişlik:", default_recommended_width_cm, "cm")

## 4️⃣ AI görsellerini yükle

AI ile oluşturduğun **son dosyaları** burada seç.

Desteklenen biçimler:

- PNG
- JPG / JPEG
- WEBP

Şeffaf arka plan gerekiyorsa tercihen şeffaf PNG yükle. Notebook arka planı değiştirmez.

In [ ]:
#@title 4️⃣ AI Görsellerini Yükle
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
supported = {".png", ".jpg", ".jpeg", ".webp"}
input_files = []

for filename, data in uploaded.items():
    ext = Path(filename).suffix.lower()
    if ext not in supported:
        print(f"⚠️ Desteklenmeyen dosya atlandı: {filename}")
        continue

    destination = INPUT_DIR / Path(filename).name
    destination.write_bytes(data)
    input_files.append(destination)
    print(f"✅ Yüklendi: {destination.name}")

input_files = sorted(input_files)

if not input_files:
    raise RuntimeError("Analiz edilecek desteklenen görsel bulunamadı.")

print(f"\n📦 Toplam {len(input_files)} görsel analiz için hazır.")

## 5️⃣ Teknik analiz motoru

Aşağıdaki ölçümler yapılır:

- Boyut ve en-boy oranı
- Dosya büyüklüğü
- Şeffaf arka plan
- Dış kenar kalıntısı
- Keskinlik
- Koyu ve çok siyah alan oranı
- Çok açık alan oranı
- İnce detay yoğunluğu
- Teknik maksimum baskı genişliği
- Minimum güvenli ve önerilen baskı genişliği

In [ ]:
#@title 5️⃣ Teknik Analiz Motoru
import os, json, math
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display

def load_rgba(path):
    with Image.open(path) as image:
        return np.array(image.convert("RGBA"))

def resize_for_analysis(rgba, max_side=1800):
    h, w = rgba.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale >= 1.0:
        return rgba.copy()
    return cv2.resize(
        rgba,
        (max(1, round(w * scale)), max(1, round(h * scale))),
        interpolation=cv2.INTER_AREA
    )

def visible_mask(rgba):
    return rgba[:, :, 3] > 20

def sharpness_metric(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0
    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.var(laplacian[mask]))

def tone_metrics(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return dict(mean=0.0, dark=100.0, near_black=100.0, near_white=0.0)

    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)[mask]
    return {
        "mean": float(gray.mean()),
        "dark": float((gray < 45).mean() * 100),
        "near_black": float((gray < 8).mean() * 100),
        "near_white": float((gray > 247).mean() * 100)
    }

def detail_metric(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0
    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 70, 170)
    edges[~mask] = 0
    return float((edges > 0).sum() / max(1, mask.sum()) * 100)

def transparency_metrics(rgba):
    alpha = rgba[:, :, 3]
    h, w = alpha.shape
    border_width = max(2, int(min(h, w) * 0.01))
    border = np.concatenate([
        alpha[:border_width, :].ravel(),
        alpha[-border_width:, :].ravel(),
        alpha[:, :border_width].ravel(),
        alpha[:, -border_width:].ravel()
    ])
    return {
        "transparent_ratio": float((alpha == 0).mean() * 100),
        "semi_transparent_ratio": float(((alpha > 0) & (alpha < 245)).mean() * 100),
        "visible_border_ratio": float((border > 10).mean() * 100),
        "has_transparency": bool(np.any(alpha < 255))
    }

def print_widths(width_px, detail, dark):
    technical_max = width_px / target_dpi * 2.54

    if detail >= 27:
        minimum = 30
    elif detail >= 23:
        minimum = 28
    elif detail >= 18:
        minimum = 25
    elif detail >= 11:
        minimum = 22
    else:
        minimum = 18

    if dark >= 45:
        minimum += 2

    recommended = max(minimum + 2, default_recommended_width_cm)
    recommended = min(recommended, technical_max)

    return round(minimum, 1), round(recommended, 1), round(technical_max, 1)

def quality_decision(result):
    failures, warnings, notes = [], [], []

    if result["short_side"] < 2000:
        failures.append("Çözünürlük düşük.")
    elif result["short_side"] < 3000:
        warnings.append("Çözünürlük orta seviyede.")

    if result["sharpness"] < 30:
        failures.append("Belirgin bulanıklık riski var.")
    elif result["sharpness"] < 60:
        warnings.append("Keskinlik düşük olabilir.")

    if result["dark"] > 60:
        warnings.append("Koyu alan oranı çok yüksek.")
    elif result["dark"] > 38:
        warnings.append("Koyu alan oranı yüksek.")

    if result["near_black"] > 20:
        warnings.append("Tam siyaha yakın alanlar baskıda birleşebilir.")

    if result["detail"] > 27:
        warnings.append("Çok yüksek ince detay yoğunluğu var.")
    elif result["detail"] > 22:
        warnings.append("İnce detay yoğunluğu yüksek.")

    if result["file_mb"] > max_file_size_mb:
        failures.append("Dosya boyutu belirlenen sınırı aşıyor.")

    if not result["has_transparency"]:
        notes.append("Dosyada şeffaflık bulunmuyor.")
    elif result["visible_border"] > 5:
        notes.append("Dış kenarlarda görünür pikseller var; tasarımın doğal parçası olabilir.")

    if failures:
        score = max(40, 72 - len(failures) * 15 - len(warnings) * 4)
        return "FAIL", "❌ YENİDEN İŞLE", score, failures, warnings, notes
    if len(warnings) >= 3:
        return "CHECK", "⚠️ KONTROL ET", 74, failures, warnings, notes
    if len(warnings) == 2:
        return "READY_CHECK", "✅ BASKIYA UYGUN — KÜÇÜK KONTROL", 86, failures, warnings, notes
    if len(warnings) == 1:
        return "READY_CHECK", "✅ BASKIYA UYGUN — KÜÇÜK KONTROL", 94, failures, warnings, notes
    return "READY", "✅ BASKIYA HAZIR", 100, failures, warnings, notes

def analyze_image(path):
    full = load_rgba(path)
    sample = resize_for_analysis(full)
    h, w = full.shape[:2]

    tones = tone_metrics(sample)
    trans = transparency_metrics(sample)
    detail = detail_metric(sample)
    sharpness = sharpness_metric(sample)
    file_mb = os.path.getsize(path) / (1024 ** 2)
    minimum_cm, recommended_cm, technical_max_cm = print_widths(w, detail, tones["dark"])

    result = {
        "file": str(path),
        "width": int(w),
        "height": int(h),
        "short_side": int(min(w, h)),
        "aspect_ratio": round(w / h, 3),
        "file_mb": round(float(file_mb), 2),
        "sharpness": round(float(sharpness), 1),
        "mean_brightness": round(tones["mean"], 1),
        "dark": round(tones["dark"], 1),
        "near_black": round(tones["near_black"], 1),
        "near_white": round(tones["near_white"], 1),
        "detail": round(float(detail), 1),
        "transparent": round(trans["transparent_ratio"], 2),
        "semi_transparent": round(trans["semi_transparent_ratio"], 2),
        "visible_border": round(trans["visible_border_ratio"], 2),
        "has_transparency": trans["has_transparency"],
        "minimum_print_cm": minimum_cm,
        "recommended_print_cm": recommended_cm,
        "technical_max_cm": technical_max_cm
    }

    code, decision, score, failures, warnings, notes = quality_decision(result)
    result.update({
        "code": code,
        "decision": decision,
        "score": score,
        "failures": failures,
        "warnings": warnings,
        "notes": notes
    })
    return result

print("✅ Teknik analiz motoru hazır.")

## 6️⃣ Baskıya uygunluk analizi

Her görsel için teknik karar, puan ve anlaşılır açıklama gösterilir.

In [ ]:
#@title 6️⃣ Baskıya Uygunluk Analizi
analysis_results = []

for path in input_files:
    result = analyze_image(path)
    analysis_results.append(result)

    print("\n" + "=" * 82)
    print("🐺 WOLFPICTURE — BASKI ANALİZİ")
    print("=" * 82)
    print(f"📄 Görsel: {path.name}")
    print(f"📐 Boyut: {result['width']} × {result['height']} px")
    print(f"📦 Dosya: {result['file_mb']:.2f} MB")
    print(f"🎯 KARAR: {result['decision']}")
    print(f"📊 Teknik puan: {result['score']}/100")
    print(f"🔍 Keskinlik: {result['sharpness']}")
    print(f"🌑 Koyu alan: %{result['dark']}")
    print(f"⚫ Tam siyaha yakın: %{result['near_black']}")
    print(f"🧵 Detay yoğunluğu: %{result['detail']}")
    print(f"🫥 Şeffaf alan: %{result['transparent']}")
    print(f"🧼 Dış kenar görünür piksel: %{result['visible_border']}")

    if result["failures"]:
        print("\n❌ DÜZELTİLMESİ GEREKENLER")
        for item in result["failures"]:
            print("•", item)

    if result["warnings"]:
        print("\n⚠️ KONTROL NOKTALARI")
        for item in result["warnings"]:
            print("•", item)

    if result["notes"]:
        print("\nℹ️ BİLGİ NOTLARI")
        for item in result["notes"]:
            print("•", item)

## 7️⃣ Tişört rengi önerisi

Tasarım farklı kumaş renklerinde matematiksel olarak simüle edilir.

Program:

- En iyi görünen renkleri
- Kullanılabilir renkleri
- Riskli renkleri

puanlayarak gösterir.

In [ ]:
#@title 7️⃣ Tişört Rengi Önerisi
GARMENT_COLORS = {
    "Beige": (222, 205, 170),
    "Black": (18, 18, 18),
    "Blue": (30, 90, 180),
    "Bronze": (205, 127, 50),
    "Brown": (100, 65, 40),
    "Clear": (245, 245, 245),
    "Copper": (184, 115, 51),
    "Gold": (212, 175, 55),
    "Gray": (128, 128, 128),
    "Green": (45, 130, 70),
    "Orange": (240, 120, 30),
    "Pink": (235, 130, 170),
    "Purple": (120, 70, 150),
    "Rainbow": (128, 128, 128),
    "Red": (200, 40, 45),
    "Rose gold": (183, 110, 121),
    "Silver": (192, 192, 192),
    "White": (255, 255, 255),
    "Yellow": (245, 210, 45)
}

def contrast_score(rgba, background_rgb):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0

    rgb = rgba[:, :, :3].astype(np.float32)
    alpha = rgba[:, :, 3].astype(np.float32) / 255.0
    background = np.empty_like(rgb)
    background[:] = background_rgb
    composite = rgb * alpha[..., None] + background * (1 - alpha[..., None])

    gray = cv2.cvtColor(composite.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    bg_gray = 0.299 * background_rgb[0] + 0.587 * background_rgb[1] + 0.114 * background_rgb[2]
    difference = np.abs(gray.astype(np.float32) - bg_gray)

    mean_difference = float(np.mean(difference[mask]))
    lost_ratio = float((difference[mask] < 24).mean() * 100)
    score = mean_difference / 1.15 - lost_ratio * 0.40

    return round(max(0.0, min(100.0, score)), 1)

color_results = {}

for path in input_files:
    rgba = resize_for_analysis(load_rgba(path))
    ranking = sorted(
        [(name, contrast_score(rgba, rgb)) for name, rgb in GARMENT_COLORS.items()],
        key=lambda item: item[1],
        reverse=True
    )
    color_results[path.name] = ranking

    print("\n" + "=" * 82)
    print(f"👕 {path.name}")

    print("\n🏆 EN İYİ RENKLER")
    for name, score in ranking[:4]:
        print(f"✅ {name}: {score}/100")

    print("\n🟡 KULLANILABİLİR RENKLER")
    for name, score in ranking[4:8]:
        print(f"• {name}: {score}/100")

    print("\n🚫 EN RİSKLİ RENKLER")
    for name, score in ranking[-3:]:
        print(f"⚠️ {name}: {score}/100")

## 8️⃣ Baskı boyu önerisi

Program, piksel boyutu ve detay yoğunluğuna göre şunları verir:

- Minimum güvenli baskı genişliği
- Önerilen baskı genişliği
- 300 DPI teknik maksimum genişlik
- Küçük göğüs baskısı veya büyük ön baskı önerisi

> Son yerleşim seçilen Printify ürününün gerçek baskı alanı içinde yapılmalıdır.

In [ ]:
#@title 8️⃣ Baskı Boyu Önerisi
for result in analysis_results:
    print("\n" + "=" * 82)
    print(f"📄 {Path(result['file']).name}")
    print(f"📏 Minimum güvenli genişlik: {result['minimum_print_cm']} cm")
    print(f"✨ Önerilen genişlik: {result['recommended_print_cm']} cm")
    print(f"🖨️ 300 DPI teknik maksimum: {result['technical_max_cm']} cm")

    if result["detail"] >= 27:
        print("🚫 Küçük sol göğüs baskısı önerilmez.")
        print("✅ Büyük ön göğüs baskısı önerilir.")
    elif result["detail"] >= 18:
        print("ℹ️ Orta veya büyük ön baskı önerilir.")
    else:
        print("✅ Küçük, orta veya büyük yerleşime daha uygundur.")

    if result["dark"] >= 38:
        print("👕 Açık renk tişörtler öncelikli değerlendirilmelidir.")

## 9️⃣ Tişört mockupları

Her tasarım için en yüksek puan alan tişört renklerinde sade mockuplar oluşturulur.

Bunlar:

- Renk uyumunu
- Tasarımın yaklaşık büyüklüğünü
- Açık ve koyu kumaştaki görünürlüğünü

kontrol etmek içindir. Orijinal tasarım dosyası değiştirilmez.

In [ ]:
#@title 9️⃣ Tişört Mockupları Oluştur
def make_mockup(design_path, garment_rgb, garment_name, output_path, width_cm):
    canvas_w, canvas_h = 1600, 1800
    canvas = Image.new("RGB", (canvas_w, canvas_h), (238, 238, 238))
    draw = ImageDraw.Draw(canvas)

    brightness = sum(garment_rgb) / 3
    outline = (55, 55, 55) if brightness > 140 else (180, 180, 180)

    shirt = [
        (420, 190), (610, 95), (755, 165), (845, 165), (990, 95), (1180, 190),
        (1450, 520), (1225, 685), (1160, 525), (1160, 1600),
        (440, 1600), (440, 525), (375, 685), (150, 520)
    ]
    draw.polygon(shirt, fill=garment_rgb, outline=outline)

    with Image.open(design_path) as image:
        design = image.convert("RGBA")
        size_ratio = min(1.0, max(0.55, width_cm / 36.0))
        max_w = int(760 * size_ratio)
        max_h = int(950 * size_ratio)
        design.thumbnail((max_w, max_h), Image.Resampling.LANCZOS)

    x = (canvas_w - design.width) // 2
    y = 370 + (950 - design.height) // 4
    canvas.paste(design, (x, y), design)

    draw.rounded_rectangle(
        (55, 55, canvas_w - 55, canvas_h - 55),
        radius=32, outline=(125, 125, 125), width=4
    )
    label_color = (40, 40, 40)
    draw.text((90, 1660), f"{garment_name} tişört", fill=label_color)
    draw.text((90, 1705), f"Önerilen baskı genişliği: {width_cm} cm", fill=label_color)

    canvas.save(output_path, quality=95)

mockup_files = []

if create_mockups:
    for path, result in zip(input_files, analysis_results):
        ranking = color_results[path.name][:int(mockup_color_count)]

        for garment_name, score in ranking:
            output = MOCKUP_DIR / (
                f"{path.stem}_{garment_name.replace(' ', '_')}_{score}.jpg"
            )
            make_mockup(
                path,
                GARMENT_COLORS[garment_name],
                garment_name,
                output,
                result["recommended_print_cm"]
            )
            mockup_files.append(output)

    print(f"✅ {len(mockup_files)} mockup oluşturuldu.\n")

    for output in mockup_files:
        display(Image.open(output).resize((280, 315)))
else:
    print("ℹ️ Mockup oluşturma kapalı.")

## 🔟 Son kalite özeti

Bütün sonuçlar tek bir profesyonel özet halinde gösterilir.

In [ ]:
#@title 🔟 Son Kalite Özeti
summary_rows = []

for result in analysis_results:
    filename = Path(result["file"]).name
    ranking = color_results.get(filename, [])
    best_colors = ", ".join(name for name, _ in ranking[:4])
    risky_colors = ", ".join(name for name, _ in ranking[-3:])

    summary_rows.append({
        "Dosya": filename,
        "Karar": result["decision"],
        "Puan": result["score"],
        "Boyut": f"{result['width']}×{result['height']}",
        "Minimum Baskı cm": result["minimum_print_cm"],
        "Önerilen Baskı cm": result["recommended_print_cm"],
        "En İyi Tişört Renkleri": best_colors,
        "Riskli Renkler": risky_colors,
        "Şeffaf Arka Plan": "Evet" if result["has_transparency"] else "Hayır",
        "Dosya MB": result["file_mb"]
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

ready = sum(row["Karar"].startswith("✅ BASKIYA HAZIR") for row in summary_rows)
ready_check = sum("KÜÇÜK KONTROL" in row["Karar"] for row in summary_rows)
check = sum(row["Karar"].startswith("⚠️") for row in summary_rows)
fail = sum(row["Karar"].startswith("❌") for row in summary_rows)

print("\n" + "=" * 82)
print("📋 TOPLU SONUÇ")
print("=" * 82)
print("✅ Baskıya hazır:", ready)
print("✅ Küçük kontrolle uygun:", ready_check)
print("⚠️ Kontrol gerekli:", check)
print("❌ Yeniden hazırlanmalı:", fail)
print("📊 Toplam görsel:", len(summary_rows))

## 1️⃣1️⃣ CSV, JSON ve ZIP indir

Raporlar ile mockuplar tek paket halinde hazırlanır.

ZIP dosyası şunları içerir:

- CSV raporu
- JSON raporu
- Mockuplar
- Yüklenen orijinal görsellerin kopyaları

> Notebook orijinal görsellerin içeriğini değiştirmez.

In [ ]:
#@title 1️⃣1️⃣ Raporları ve ZIP Paketini İndir
from google.colab import files
import zipfile, shutil

csv_path = REPORT_DIR / "WolfPicture_Printify_Analysis.csv"
json_path = REPORT_DIR / "WolfPicture_Printify_Analysis.json"
zip_path = Path("/content/WolfPicture_Printify_Analysis_Package.zip")

summary_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
json_path.write_text(
    json.dumps(summary_rows, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in input_files:
        archive.write(path, Path("original_images") / path.name)

    for path in mockup_files:
        archive.write(path, Path("mockups") / path.name)

    archive.write(csv_path, Path("reports") / csv_path.name)
    archive.write(json_path, Path("reports") / json_path.name)

print("✅ CSV hazır:", csv_path.name)
print("✅ JSON hazır:", json_path.name)
print("✅ ZIP hazır:", zip_path.name)

files.download(str(zip_path))

# ✅ Tamamlandı

Bu son sürümde işlem sırası şöyledir:

**AI görselini hazırla → görseli yükle → teknik analiz → baskı kararı → tişört renkleri → baskı boyu → mockuplar → son kalite özeti → CSV/JSON/ZIP**

Arka plan silme ve upscale işlemleri bu notebookta bulunmaz. Böylece görselin değiştirilmeden, yalnızca baskı açısından değerlendirilir.